# 03-2 규칙 고친 뒤 표·검색 목록 다시 계산
**왜:** 서준영 정답지 채점(③)에서 놓침의 대부분이 **baklava 밀, tiramisu 밀**이었다. 정답 음식을 알아도 놓친 칸 → 사진 AI가 아니라 **재료 규칙 누락**.
`knowledge/rules.py`의 밀 규칙에 phyllo/filo, ladyfingers 등이 없어서 표에 baklava 밀 11%, tiramisu 밀 11% → `해당 없음`으로 나왔다.

**고친 것 (`knowledge/rules.py`, 2026-11-17)**
- 밀: phyllo·filo, ladyfingers·savoiardi, pastry, dough, crust, crouton, biscuit, cookie, graham, cake, tortilla(옥수수 제외), pita, naan, bagel, croissant, wonton, seitan, spelt, farro, 파스타 모양 이름(gnocchi, orzo, lasagna, ravioli …), udon. 제외에 rice cake, corn tortilla 추가
- 알류: ladyfingers·savoiardi (달걀 반죽)
- 우유: mascarpone, ricotta, mozzarella, parmesan, half-and-half (이름에 cheese·cream이 없는 유제품)

**이 노트북이 하는 일** (1M_Final 1.4GB를 다시 읽지 않고, 03에서 뽑아 둔 `table_recipes.parquet`만 씀)
1. 예전 규칙으로 다시 계산해 지금 표(`food_table.csv`)와 **같은지 확인** (같아야 이 방법이 맞음)
2. 새 규칙으로 175종 × 19항목 비율 다시 계산 → 바뀐 음식·항목 표 → `찍먹_실험기록/표/food_table_v2.csv` (영양·레시피 수는 그대로)
3. 2단계 검색 목록 `best/clip_base/index_meta.parquet`의 19항목도 새 규칙으로 (글 벡터 `index_emb.npy`는 그대로). 예전 파일은 `index_meta_v1.parquet`로 남김

끝나면 Claude에게 "03-2 다 돌렸어" → `food_table_v2.csv`를 깃 `knowledge/food_table.csv`에 반영. 앱을 돌리는 사람(유라님)은 `clip_base/index_meta.parquet`를 다시 복사.

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')
SAVE = f'{EXP}/best/clip_base'   # 앱 검색 목록 (기본 CLIP)
os.makedirs(SAVE, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
print('준비 완료 (CPU로 충분, 5분 안팎)')

## 1. 예전 규칙으로 다시 계산 → 지금 표와 같은지 확인

In [ ]:
import re, shutil
import numpy as np, pandas as pd
import knowledge.rules as R
from knowledge.rules import allergen_flags, ingredients_text

subprocess.run([sys.executable, '-m', 'knowledge.rules'], cwd=REPO, check=True)   # 새 규칙 자체 검사

# 2026-11-17 이전 규칙 (확인용으로만)
OLD = dict(R.RULES)
OLD['알류'] = ('(?<![a-z])eggs?(?![a-z])|mayonnaise|meringue', '')
OLD['우유'] = ('milk|cream|butter|cheese|yogurt|yoghurt|whey|ghee|custard', R.RULES['우유'][1])
OLD['밀'] = ('wheat|flour|bread|pasta|spaghetti|macaroni|couscous|noodle|cracker|panko|semolina|bulgur|soy sauce',
            'rice flour|rice noodle|corn flour|cornflour|almond flour|coconut flour|chickpea flour|buckwheat flour|gluten-free|gluten free')
NEW = dict(R.RULES)

def flags_with(rules, text):
    R.RULES = rules
    try:
        return allergen_flags(text)
    finally:
        R.RULES = NEW

recipes = pd.read_parquet(RECIPES, columns=['recipe_idx', 'label', 'NER']).drop_duplicates(['label', 'recipe_idx']).reset_index(drop=True)
recipes['ing'] = recipes.NER.map(ingredients_text)
table = pd.read_csv(f'{REPO}/knowledge/food_table.csv', encoding='utf-8-sig').set_index('class')
A = list(NEW)
ratio = lambda fl: fl.set_index(recipes.label.values).groupby(level=0).mean().reindex(table.index).fillna(0)   # build_table과 같게

old = ratio(flags_with(OLD, recipes.ing))
diff = (old[A] - table[A]).abs()
print(f'레시피 {len(recipes):,}개, 음식 {recipes.label.nunique()}종')
print('예전 규칙 재계산 vs 지금 표: 최대 차이', round(float(diff.max().max()), 4), '/ 표에 없는 음식', int(old[A].isna().any(axis=1).sum()))
assert diff.max().max() < 0.002, '예전 규칙으로 계산한 값이 지금 표와 다름 → table_recipes가 표 만들 때와 다름. 멈추고 Claude에게 출력 공유'

## 2. 새 규칙으로 표 다시 계산

In [ ]:
new_fl = flags_with(NEW, recipes.ing)
new = ratio(new_fl).round(4)
v2 = table.copy()
v2[A] = new[A]
chg = (v2[A] - table[A])
lv = lambda v: np.where(v >= 0.6, '주의', np.where(v >= 0.2, '확인 필요', '해당 없음'))
moved = [(c, a, round(float(table.loc[c, a]), 3), round(float(v2.loc[c, a]), 3), lv(table.loc[c, a]).item(), lv(v2.loc[c, a]).item())
         for c in table.index for a in A if lv(table.loc[c, a]) != lv(v2.loc[c, a])]
moved = pd.DataFrame(moved, columns=['음식', '항목', '전', '후', '전 단계', '후 단계'])
print('항목별 비율이 바뀐 음식 수:', (chg.abs() > 0.001).sum()[lambda s: s > 0].to_dict())
print(f'단계가 바뀐 칸 {len(moved)}개 (175종 × 19항목 중)')
display(moved.sort_values(['항목', '후'], ascending=[True, False]).reset_index(drop=True))
display(v2.loc[['baklava', 'tiramisu', 'apple_pie', 'cheesecake', 'lasagna', 'gnocchi', 'pho', 'pad_thai'], ['밀', '알류', '우유']]
        .join(table[['밀', '알류', '우유']], rsuffix=' (전)'))

out = f'{EXP}/표/food_table_v2.csv'
v2.reset_index().to_csv(out, index=False, encoding='utf-8-sig')
moved.to_csv(f'{EXP}/표/03-2_단계바뀐칸.csv', index=False, encoding='utf-8-sig')
print('저장:', out)

## 3. 2단계 검색 목록 19항목 갱신

In [ ]:
IDX = f'{EXP}/best/clip_base'
meta = pd.read_parquet(f'{IDX}/index_meta.parquet')
if not os.path.exists(f'{IDX}/index_meta_v1.parquet'):
    shutil.copy(f'{IDX}/index_meta.parquet', f'{IDX}/index_meta_v1.parquet')      # 예전 것 남김
m = meta[['recipe_idx', 'label']].merge(recipes[['recipe_idx', 'label', 'ing']], on=['recipe_idx', 'label'], how='left')
assert len(m) == len(meta) and m.ing.notna().all(), '검색 목록 레시피를 table_recipes에서 못 찾음'
fl = flags_with(NEW, m.ing).astype('uint8')
before = meta[A].mean()
meta[A] = fl[A].values
meta.to_parquet(f'{IDX}/index_meta.parquet', index=False)
print(f'검색 목록 {len(meta):,}줄 19항목 갱신 → {IDX}/index_meta.parquet')
display(pd.DataFrame({'전': before, '후': meta[A].mean()}).round(3).loc[['밀', '알류', '우유']])

## 끝
- `단계가 바뀐 칸` 표가 그럴듯한지 본다 (예: baklava·tiramisu 밀이 `주의`로, 쌀국수(pho)는 그대로 `해당 없음`).
- Claude에게 **"03-2 다 돌렸어"** → 드라이브 `표/food_table_v2.csv`를 깃 `knowledge/food_table.csv`에 반영하고, 서준영 ③ 채점(`evaluation/merge_eval.py`)을 새 표로 다시 낸다.
- 앱을 돌리는 컴퓨터는 `best/clip_base/index_meta.parquet`를 다시 `knowledge/index/`에 복사.
- 되돌리기: `index_meta_v1.parquet` → `index_meta.parquet`로 이름 바꾸기.